# PHYT1D — Module 07: MCMC Window B — Exercise Parameter Identification

Identifies 6 exercise-specific parameters from **exercise-window CGM**
(exercise bout + 6 h post-exercise), conditioned on Window A posteriors.

θ_B = {beta_aer, beta_res, tau_post, tau_on, VO2max, phi}

**Key novel parameter:** tau_post — post-exercise SI decay time constant  
**Pass criterion:** MARD < 10%, VO2max recovery < 15%, tau_post agrees with T1DS IS decay


In [ ]:
import numpy as np
from scipy import stats


## 7.1 Log-Prior for Group B

In [ ]:
def log_prior_B(theta_B, age_group="adult", sex="M"):
    """
    Log prior for Group B exercise parameters.

    Parameters
    ----------
    theta_B    : dict — {beta_aer, beta_res, tau_post, tau_on, VO2max, phi}
    age_group  : str  — for VO2max prior mean
    sex        : str  — 'M' | 'F'

    Returns
    -------
    float — log p(θ_B)
    """
    AGE = {"child":10,"adolescent":15,"adult":35}
    age = AGE.get(age_group, 35)
    vo2_mean = (-0.0010*age**2 - 0.2248*age + 54.286 if sex.upper()=="M"
                else -0.0021*age**2 - 0.1407*age + 43.066)

    priors = {
        "beta_aer": ("normal",  0.008, 0.003),
        "beta_res": ("normal",  0.005, 0.002),
        "tau_post": ("gamma",   4.0,   60.0),    # shape=4, scale=60 → mean~240
        "tau_on"  : ("normal",  15.0,  5.0),
        "VO2max"  : ("normal",  vo2_mean, 8.0),
        "phi"     : ("uniform", 0.0,   0.5),
    }

    lp = 0.0
    for name, val in theta_B.items():
        kind = priors[name][0]
        if kind == "normal":
            mu, sigma = priors[name][1], priors[name][2]
            lp += stats.norm.logpdf(val, mu, sigma)
        elif kind == "gamma":
            shape, scale = priors[name][1], priors[name][2]
            if val <= 0: return -np.inf
            lp += stats.gamma.logpdf(val, a=shape, scale=scale)
        elif kind == "uniform":
            lo, hi = priors[name][1], priors[name][2]
            if not (lo <= val <= hi): return -np.inf

    # Positivity constraints
    for name in ["beta_aer","beta_res","tau_post","tau_on","VO2max"]:
        if theta_B.get(name, 1.0) <= 0:
            return -np.inf

    return lp


## 7.2 Window B Log-Likelihood

In [ ]:
def log_likelihood_B(theta_B, theta_A_median, user_params, fixed,
                     cgm_times_ex, cgm_obs_ex, sigma_cgm=5.0):
    """
    Gaussian log-likelihood on exercise-window CGM residuals.
    θ_A is fixed at Window A posterior medians.

    Parameters
    ----------
    theta_B        : dict — Group B params (being identified)
    theta_A_median : dict — Group A posterior medians (fixed)
    user_params    : dict — Group C (exercise prescription, meals)
    fixed          : dict — Group D
    cgm_times_ex   : array — CGM times in exercise window [min]
    cgm_obs_ex     : array — observed CGM in exercise window [mg/dL]
    sigma_cgm      : float — CGM noise σ [mg/dL]

    Returns
    -------
    float — log L
    """
    from 06_simulator import simulate_trajectory
    try:
        times, ig_sim, _ = simulate_trajectory(
            theta_A_median, theta_B, user_params, fixed)
        # Interpolate sim to CGM times
        ig_at_cgm = np.interp(cgm_times_ex, times, ig_sim)
        residuals  = cgm_obs_ex - ig_at_cgm
        ll = -0.5 * np.sum((residuals / sigma_cgm)**2)
        ll -= len(cgm_obs_ex) * np.log(sigma_cgm * np.sqrt(2 * np.pi))
        return ll
    except Exception:
        return -np.inf


## 7.3 Adaptive SCMH for Window B

In [ ]:
def run_MCMC_window_B(theta_B_init, theta_A_median, user_params, fixed,
                      cgm_times_ex, cgm_obs_ex, age_group="adult", sex="M",
                      n_samples=2000, burn_in=1000, rng=None, verbose=True):
    """
    Run MCMC Window B — exercise parameter identification.

    Uses the same adaptive SCMH algorithm as Window A, but for θ_B.
    θ_A is conditioned on Window A posterior medians (fixed).

    Returns
    -------
    chain_B      : list of dicts — posterior samples for θ_B
    accept_rates : dict
    """
    if rng is None:
        rng = np.random.default_rng(42)

    param_names = list(theta_B_init.keys())
    theta_cur   = {k: v for k, v in theta_B_init.items()}

    def log_post(theta):
        lp = log_prior_B(theta, age_group, sex)
        if not np.isfinite(lp): return -np.inf
        ll = log_likelihood_B(theta, theta_A_median, user_params, fixed,
                              cgm_times_ex, cgm_obs_ex)
        return lp + ll

    lp_cur    = log_post(theta_cur)
    sigma_prop = {k: abs(v)*0.10 + 1e-6 for k, v in theta_cur.items()}
    accept_cnt = {k: 0 for k in param_names}
    total_cnt  = {k: 0 for k in param_names}
    chain_B    = []

    for i in range(n_samples):
        for name in param_names:
            theta_prop = {k: v for k, v in theta_cur.items()}
            theta_prop[name] = theta_cur[name] + rng.normal(0, sigma_prop[name])

            lp_prop    = log_post(theta_prop)
            log_alpha  = lp_prop - lp_cur

            if np.log(rng.uniform()) < log_alpha:
                theta_cur = theta_prop
                lp_cur    = lp_prop
                accept_cnt[name] += 1
            total_cnt[name] += 1

        if i < burn_in and (i+1) % 50 == 0:
            for name in param_names:
                rate = accept_cnt[name] / max(total_cnt[name], 1)
                if rate > 0.28:  sigma_prop[name] *= 1.1
                elif rate < 0.18: sigma_prop[name] *= 0.9

        if i >= burn_in:
            chain_B.append({k: v for k, v in theta_cur.items()})

        if verbose and (i+1) % 500 == 0:
            avg = np.mean([accept_cnt[k]/max(total_cnt[k],1) for k in param_names])
            print(f"  Iter {i+1:>5d}/{n_samples}  avg accept={avg:.2f}  lp={lp_cur:.2f}")

    accept_rates = {k: accept_cnt[k]/max(total_cnt[k],1) for k in param_names}
    return chain_B, accept_rates


## 7.4 VO2max Recovery Evaluation

In [ ]:
def evaluate_VO2max_recovery(chain_B, VO2max_true, threshold=0.15):
    """
    VO2max recovery: |VO2max_hat − VO2max_true| / VO2max_true < 15%

    Parameters
    ----------
    chain_B      : list of dicts
    VO2max_true  : float — ground-truth VO2max [mL/kg/min]
    threshold    : float — pass threshold (default 0.15 = 15%)

    Returns
    -------
    dict — {hat, error_pct, passed}
    """
    vo2_samples = [s["VO2max"] for s in chain_B]
    vo2_hat     = np.median(vo2_samples)
    err_pct     = abs(vo2_hat - VO2max_true) / VO2max_true * 100
    passed      = err_pct < threshold * 100

    print(f"\n── VO2max Recovery ─────────────────────────────────────────")
    print(f"  True = {VO2max_true:.1f}  Est = {vo2_hat:.1f}  Error = {err_pct:.1f}%  "
          f"{'PASS ✓' if passed else 'FAIL ✗'}")
    return {"hat": vo2_hat, "error_pct": err_pct, "passed": passed}


def evaluate_tau_post(chain_B, tau_post_implied=None):
    """
    tau_post recovery: compare posterior median to T1DS IS controller implied decay.

    Parameters
    ----------
    chain_B         : list of dicts
    tau_post_implied: float | None — implied τ_post from T1DS IS controller [min]

    Returns
    -------
    dict — {median, 25p, 75p, relative_error}
    """
    tp_samples = [s["tau_post"] for s in chain_B]
    tp_50 = np.percentile(tp_samples, 50)
    tp_25 = np.percentile(tp_samples, 25)
    tp_75 = np.percentile(tp_samples, 75)

    print(f"\n── tau_post Recovery ───────────────────────────────────────")
    print(f"  Posterior: median={tp_50:.1f}  [25th={tp_25:.1f}, 75th={tp_75:.1f}] min")
    if tau_post_implied is not None:
        err = abs(tp_50 - tau_post_implied) / tau_post_implied * 100
        print(f"  T1DS implied = {tau_post_implied:.1f}  Relative error = {err:.1f}%")
        return {"median":tp_50,"25p":tp_25,"75p":tp_75,"relative_error":err}
    return {"median":tp_50,"25p":tp_25,"75p":tp_75}
